"DO ecology extraction" is a notebook I made years ago to get bottom-layer dissolved oxygen out of the ecology-provided netCDF files of their 2021 model results. But it is very slow.

"DO ecology extraction v2" makes some changes to how results are pulled from the file to improve performance. It runs very fast now. But first we need to QA that it reshaped the data correctly. To do that I'm comparing outputs from the original version to the new one.

In [12]:
do_nc = "/home/benr/src/ssm-analysis/model_results/bottomdo_bounding2021_yr2014_existonly.nc"
do_nc_v2 = "/home/benr/src/ssm-analysis/model_results/bottomdo_bounding2021_yr2014_existonly_v2.nc"

import random
from netCDF4 import Dataset
import numpy as np

In [13]:
ds1 = Dataset(do_nc)
ds2 = Dataset(do_nc_v2)
ds1

<class 'netCDF4.Dataset'>
root group (NETCDF4 data model, file format HDF5):
    model_start: 2014.01.01
    dimensions(sizes): time(8760), node(16012)
    variables(dimensions): int32 node(node), int32 h(node), int32 x(node), int32 y(node), float32 time(time), float32 existingDOXG_bottom(time, node), float32 referenceDOXG_bottom(time, node)
    groups: 

In [14]:
ds2

<class 'netCDF4.Dataset'>
root group (NETCDF4 data model, file format HDF5):
    model_start: 2014.01.01
    dimensions(sizes): time(8760), node(4155)
    variables(dimensions): int32 node(node), int32 h(node), int32 x(node), int32 y(node), float32 time(time), float32 existingDOXG_bottom(time, node)
    groups: 

The only difference in the metadata is the extra "reference" variable that got added to v1 by mistake. It takes too long to redo it, and this shouldn't matter.

Now let's spot-check a bunch of cells. If there are no errors then the QA passed.

In [15]:
random.seed()
for i in range(100):
    t = random.randrange(ds1.dimensions['time'].size)
    assert ds1['time'][t] == ds2['time'][t], f'Time index {t} does not match'
    n = random.randrange(ds1.dimensions['node'].size)
    assert ds1['node'][n] == ds2['node'][n], f'Node index {n} does not match'
    assert ds1['existingDOXG_bottom'][t,n] == ds2['existingDOXG_bottom'][t,n], f'DOXG value at ({t},{n}) does not match'

IndexError: index exceeds dimension bounds

In [16]:
ds1.close()
ds2.close()